# Analyzing a Tindeq Session — Force Curve Under the Microscope

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/8cH9azbsFifZ/climbing-science/main?labpath=notebooks%2F05_session_deep_dive.ipynb)

## Introduction

A single Tindeq Progressor session file contains a high-resolution
force–time trace — typically sampled at 80 Hz. From this raw signal
we can extract a surprising amount of information:

| Metric | What it tells you |
|---|---|
| **MVC-7** | Maximum voluntary contraction over 7 s — the gold-standard finger-strength measure |
| **Peak force** | Absolute maximum force in the session |
| **Rate of Force Development (RFD)** | How quickly you recruit motor units — critical for dynamic moves |
| **Impulse** | Total mechanical work done (area under the curve) |
| **Repeater segmentation** | Per-rep breakdown for endurance protocols |

This notebook walks through each step, from loading raw data to a
full session summary. It works **out of the box** with synthetic
demo data — just drop in your own `.json` file when you're ready.

## Setup

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## Load Your Session

**Edit the cell below** — point `SESSION_FILE` at your Tindeq JSON
export. If the file doesn't exist, the notebook generates a
realistic synthetic MVC-7 session so every cell runs cleanly.

In [ ]:
# ── CONFIGURATION (edit here) ────────────────────────────────
SESSION_FILE = "path/to/your/session.json"   # ← Change this!
# ─────────────────────────────────────────────────────────────

import os

USE_REAL_DATA = os.path.isfile(SESSION_FILE)

if USE_REAL_DATA:
    from climbing_science.adapters.tindeq import load, extract_mvc7, extract_peaks
    session = load(SESSION_FILE)
    values = session.values
    sample_rate_hz = session.sample_rate_hz
    print(f"Loaded {session.filename}  —  "
          f"{len(values)} samples @ {sample_rate_hz} Hz  "
          f"({len(values)/sample_rate_hz:.1f} s)")
else:
    # ── Synthetic MVC-7 session ──────────────────────────────
    # 2 s baseline → 7 s hang at ~40 kg → 2 s baseline
    sample_rate_hz = 80
    rng = np.random.default_rng(42)

    baseline_pre  = rng.normal(loc=0.3, scale=0.2, size=2 * sample_rate_hz).clip(0)
    ramp_up       = np.linspace(0.3, 40.0, int(0.4 * sample_rate_hz))
    hang          = rng.normal(loc=40.0, scale=1.0, size=7 * sample_rate_hz)
    ramp_down     = np.linspace(40.0, 0.3, int(0.3 * sample_rate_hz))
    baseline_post = rng.normal(loc=0.3, scale=0.2, size=2 * sample_rate_hz).clip(0)

    values = list(np.concatenate([
        baseline_pre, ramp_up, hang, ramp_down, baseline_post
    ]))
    print(f"Using synthetic data  —  "
          f"{len(values)} samples @ {sample_rate_hz} Hz  "
          f"({len(values)/sample_rate_hz:.1f} s)")

## Plot Raw Force Curve

First look: the complete force–time trace. Time on the x-axis,
force (kg) on the y-axis.

In [ ]:
try:
    from climbing_science.frontends.notebook import plot_force_session
    plot_force_session(values, sample_rate_hz)
except Exception:
    # Manual fallback
    time_s = np.arange(len(values)) / sample_rate_hz
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(time_s, values, linewidth=0.8, color="steelblue")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Force [kg]")
    ax.set_title("Raw Force–Time Trace")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Smooth the Signal

Raw force data from a load cell is noisy. A moving-average filter
removes high-frequency jitter while preserving the shape of the
force curve. We compare several window sizes to see the trade-off
between noise reduction and signal fidelity.

In [ ]:
from climbing_science.signal import smooth

time_s = np.arange(len(values)) / sample_rate_hz

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time_s, values, linewidth=0.5, alpha=0.4, label="Raw", color="grey")

for win, color in [(5, "steelblue"), (15, "darkorange"), (31, "seagreen")]:
    smoothed = smooth(values, window=win, method="moving_average")
    ax.plot(time_s, smoothed, linewidth=1.2, label=f"Window = {win}", color=color)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Force [kg]")
ax.set_title("Effect of Smoothing Window Size")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Peak Detection

`detect_peaks()` finds contiguous regions above a force threshold.
Each peak object contains start/end indices, the peak value, mean
force, and duration — everything you need for per-rep analysis.

In [ ]:
from climbing_science.signal import detect_peaks

smoothed = smooth(values, window=5)
peaks = detect_peaks(smoothed, threshold=5.0, min_distance=10)

print(f"Detected {len(peaks)} peak(s):\n")
for i, p in enumerate(peaks):
    print(f"  Peak {i+1}: {p.peak_value:.1f} kg  "
          f"(mean {p.mean_value:.1f} kg, "
          f"duration {p.duration_s:.2f} s)")

# ── Annotated plot ──
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time_s, values, linewidth=0.5, alpha=0.35, color="grey", label="Raw")
ax.plot(time_s, smoothed, linewidth=1.0, color="steelblue", label="Smoothed")

for i, p in enumerate(peaks):
    t_start = p.start_idx / sample_rate_hz
    t_end   = p.end_idx / sample_rate_hz
    ax.axvspan(t_start, t_end, alpha=0.15, color="tomato")
    ax.annotate(
        f"{p.peak_value:.1f} kg",
        xy=(p.peak_idx / sample_rate_hz, p.peak_value),
        xytext=(0, 12), textcoords="offset points",
        ha="center", fontsize=9, fontweight="bold", color="tomato",
        arrowprops=dict(arrowstyle="->", color="tomato", lw=1.2),
    )

ax.axhline(5.0, color="grey", linestyle="--", linewidth=0.8, label="Threshold (5 kg)")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Force [kg]")
ax.set_title("Peak Detection")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## MVC-7 Extraction

The **MVC-7** (7-second maximum voluntary contraction) is the single
most important metric in finger-strength testing. It's the highest
average force you can sustain over any 7-second window in the
session. We highlight that window on the force curve.

In [ ]:
from climbing_science.signal import best_n_second_average

mvc7 = best_n_second_average(values, sample_rate_hz, n_seconds=7.0)
print(f"MVC-7 = {mvc7:.1f} kg")

# Find the window that produced this average
window_samples = int(7.0 * sample_rate_hz)
arr = np.array(values)
running_avg = np.convolve(arr, np.ones(window_samples) / window_samples, mode="valid")
best_start = int(np.argmax(running_avg))
best_end   = best_start + window_samples

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time_s, values, linewidth=0.5, alpha=0.35, color="grey", label="Raw")
ax.plot(time_s, smoothed, linewidth=1.0, color="steelblue", label="Smoothed")

t0 = best_start / sample_rate_hz
t1 = best_end   / sample_rate_hz
ax.axvspan(t0, t1, alpha=0.2, color="gold", label=f"Best 7 s window")
ax.axhline(mvc7, color="goldenrod", linestyle="--", linewidth=1.2,
           label=f"MVC-7 = {mvc7:.1f} kg")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Force [kg]")
ax.set_title("MVC-7 — Best 7-Second Average")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Rate of Force Development (RFD)

RFD measures how *fast* you can produce force — it's the slope of
the force–time curve in the initial pull phase. A high RFD means
you can latch holds quickly, which is critical for dynamic moves.

Levernier et al. (2019) showed that elite climbers have
significantly higher RFD than recreational climbers, even after
normalising for peak force.

In [ ]:
from climbing_science.signal import compute_rfd

rfd = compute_rfd(values, sample_rate_hz, window_ms=200)

print(f"Peak RFD    = {rfd.peak_rfd_kg_s:.1f} kg/s  "
      f"(over {rfd.rfd_window_ms} ms window)")
print(f"Average RFD = {rfd.avg_rfd_kg_s:.1f} kg/s")

# ── Visualise the derivative ──
arr = np.array(smooth(values, window=5))
dt  = 1.0 / sample_rate_hz
rfd_trace = np.gradient(arr, dt)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(time_s, arr, color="steelblue", linewidth=1.0)
ax1.set_ylabel("Force [kg]")
ax1.set_title("Smoothed Force Curve")
ax1.grid(True, alpha=0.3)

ax2.plot(time_s, rfd_trace, color="darkorange", linewidth=0.8)
ax2.axhline(0, color="grey", linewidth=0.5)
ax2.set_xlabel("Time [s]")
ax2.set_ylabel("RFD [kg/s]")
ax2.set_title(f"Rate of Force Development  —  Peak = {rfd.peak_rfd_kg_s:.1f} kg/s")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Impulse — Total Work Done

Impulse is the integral of force over time (area under the curve,
in kg·s). It captures the *total* mechanical demand of the session
and is useful for comparing workload across sessions.

In [ ]:
from climbing_science.signal import compute_impulse

impulse = compute_impulse(values, sample_rate_hz)
print(f"Total impulse = {impulse:.1f} kg·s")

# ── Filled area plot ──
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(time_s, values, alpha=0.3, color="steelblue")
ax.plot(time_s, values, linewidth=0.6, color="steelblue")
ax.set_xlabel("Time [s]")
ax.set_ylabel("Force [kg]")
ax.set_title(f"Impulse = {impulse:.1f} kg·s  (area under curve)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Repeater Segmentation *(optional)*

If your session follows a repeater protocol (e.g. 7 s hang / 3 s
rest), `segment_repeaters()` chops the trace into individual reps.
On the synthetic single-hang data this will return just one segment
— run it on a real repeater file to see the full breakdown.

In [ ]:
from climbing_science.signal import segment_repeaters

reps = segment_repeaters(values, sample_rate_hz, hang_s=7, rest_s=3)

if len(reps) > 1:
    print(f"Segmented into {len(reps)} reps:\n")
    fig, axes = plt.subplots(1, min(len(reps), 6), figsize=(16, 4), sharey=True)
    if not hasattr(axes, "__len__"):
        axes = [axes]
    for i, (rep, ax) in enumerate(zip(reps, axes)):
        seg = values[rep.start_idx:rep.end_idx]
        t   = np.arange(len(seg)) / sample_rate_hz
        ax.plot(t, seg, color="steelblue", linewidth=0.8)
        ax.set_title(f"Rep {i+1}", fontsize=10)
        ax.set_xlabel("s")
        if i == 0:
            ax.set_ylabel("kg")
        ax.grid(True, alpha=0.3)
        print(f"  Rep {i+1}: peak {max(seg):.1f} kg, "
              f"mean {np.mean(seg):.1f} kg, "
              f"duration {len(seg)/sample_rate_hz:.2f} s")
    plt.suptitle("Repeater Segmentation — Per-Rep View", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("Single effort detected (not a repeater session).\n"
          "→ Load a repeater file to see per-rep segmentation.")

## Session Summary

All metrics at a glance.

In [ ]:
duration_s = len(values) / sample_rate_hz
peak_force = max(values)

print("┌──────────────────────────────────────────┐")
print("│        SESSION SUMMARY                   │")
print("├──────────────────────────────────────────┤")
print(f"│  Duration         {duration_s:>8.1f} s             │")
print(f"│  Samples          {len(values):>8d}               │")
print(f"│  Sample rate      {sample_rate_hz:>8d} Hz            │")
print(f"│  Peak force       {peak_force:>8.1f} kg            │")
print(f"│  MVC-7            {mvc7:>8.1f} kg            │")
print(f"│  Peak RFD         {rfd.peak_rfd_kg_s:>8.1f} kg/s          │")
print(f"│  Avg RFD          {rfd.avg_rfd_kg_s:>8.1f} kg/s          │")
print(f"│  Impulse          {impulse:>8.1f} kg·s          │")
print(f"│  Peaks detected   {len(peaks):>8d}               │")
print(f"│  Repeater reps    {len(reps):>8d}               │")
print("└──────────────────────────────────────────┘")

## References

- **Levernier, G., Samozino, P., & Laffaye, G.** (2019).
  *Rate of force development and maximal force: reliability and
  difference across climbing level.*
  Sports Technology, 12(1), 96–105.

- **Giles, D., Chidley, J. B., Taylor, N., Sheridan, O., Machado, I.,
  Sherrington, A., ... & Mayberry, J.** (2019).
  *An all-out test to determine finger flexor critical force in rock climbers.*
  International Journal of Sports Physiology and Performance, 14(4), 516–523.

---

*Notebook generated by [climbing-science](https://github.com/8cH9azbsFifZ/climbing-science).*